# Fine Tuning a Canonical HMM

Example for applying a canonical HMM to a **multi-session dataset** and fine tuning it on the new data.

This assumes the data has already been preprocessed, source reconstructed and parcellated (see the tutorials in the root directory), so that there is a `lcmv-parc-raw.fif` file for each session.

Note, this notebook requires osl-dynamics v3.8.1 or later (two-stage sign flipping).

#### Why load all the sessions together?

Loading all the sessions of a dataset into one `Data` object has two benefits:

1. The data preparation runs in parallel, which is faster for large datasets.

2. The sign flipping is done in **two stages**: a template is built from the new sessions and aligned to the canonical (Cam-CAN) template, then each session is aligned to it. Sessions only have to match their own dataset, so this is more robust than aligning each session to the canonical template directly, particularly when the new data does not closely match the training data (e.g. a different scanner, preprocessing or population).

#### Why fine tune?

The canonical HMM can be applied as is, or fine tuned on the new dataset. Fine tuning starts from the canonical parameters and adapts the state covariances and transition probabilities to the new data, so the states keep their correspondence with the canonical networks while fitting the new data better. The variational free energy tells us how much we gained.

#### Steps

1. Load and prepare the data (two-stage sign flipping, time-delay embedding, PCA, standardisation).
2. Apply the canonical HMM as is and calculate the free energy of each session.
3. Fine tune the canonical HMM on the new dataset.
4. Inference with the fine-tuned model: state probabilities and session-specific covariances.

#### Import Packages

In [ ]:
import os
import pickle
from glob import glob

import numpy as np

from modules import hmm

from osl_dynamics.data import Data
from osl_dynamics.utils import plotting

#### Settings

In [ ]:
parcellation = "Glasser52"
n_states = 8

output_dir = "BIDS/derivatives/hmm/fine_tuned"
os.makedirs(output_dir, exist_ok=True)

## 1. Load and Prepare the Data

We pass all the parcellated sessions to one `Data` object. `prepare_data_for_canonical_hmm` then does the two-stage sign flipping, followed by time-delay embedding, PCA and standardisation.

The log reports the correlation between the template built from these sessions and the canonical template. This single number summarises how well the dataset matches the Cam-CAN training data.

In [ ]:
files = sorted(glob("BIDS/derivatives/osl/*/lcmv-parc-raw.fif"))
print(f"{len(files)} sessions")

data = Data(files, picks="misc", reject_by_annotation="omit", n_jobs=8)
data = hmm.prepare_data_for_canonical_hmm(data, parcellation=parcellation)

## 2. Apply the Canonical HMM

First, we load the canonical HMM.

In [ ]:
model = hmm.load_canonical_hmm(n_states=n_states, parcellation=parcellation)
model.summary()

Let's calculate the free energy of each session before fine tuning. Lower is better. The percentile places each session within the distribution of the Cam-CAN sessions (see `variational_free_energy.ipynb` for more on this).

`free_energy` returns a single value for whatever it is passed, so we ask for one dataset per session.

In [ ]:
datasets = model.make_dataset(data, concatenate=False)

fe_before = np.array([model.free_energy(dataset) for dataset in datasets])
percentile = hmm.free_energy_percentile(fe_before, n_states=n_states, parcellation=parcellation)

print("Free energy before fine tuning:", np.round(fe_before, 3))
print("Cam-CAN percentile:", np.round(percentile, 1))

In [ ]:
fig, ax = hmm.plot_free_energy_distribution(fe_before, n_states=n_states, parcellation=parcellation)

## 3. Fine Tune the Canonical HMM

`load_canonical_hmm` creates a model with `learn_means=False` and `learn_covariances=True` (the canonical HMMs have zero means), and the transition probability matrix is learnt by default. Fine tuning therefore updates the state covariances and transition probabilities.

We start from the canonical parameters, so a small learning rate and a couple of epochs are enough. Larger values fit the new data more closely but let the states drift further from the canonical networks.

In [ ]:
model.config.learning_rate = 0.001
model.config.n_epochs = 2

history = model.fit(data)
pickle.dump(history, open(f"{output_dir}/history.pkl", "wb"))

Let's check the training loss decreased.

In [ ]:
fig, ax = plotting.plot_line(
    [range(1, len(history["loss"]) + 1)],
    [history["loss"]],
    x_label="Epoch",
    y_label="Loss",
)

Now we recalculate the free energy of each session. It is the same data and the same model structure, so the values are directly comparable with the ones before fine tuning.

In [ ]:
fe_after = np.array([model.free_energy(dataset) for dataset in datasets])

print("Free energy after fine tuning:", np.round(fe_after, 3))
print("Change:", np.round(fe_after - fe_before, 3))

We should check the fine-tuned states still correspond to the canonical networks. Here, we correlate the off-diagonal elements of each fine-tuned state covariance with each canonical state covariance. The diagonal of this matrix should be the largest entry in each row, i.e. each state should still look most like the canonical state it started from.

In [ ]:
canonical_covs = np.load(f"models/{parcellation}/{n_states:02d}_states/covs.npy")
fine_tuned_covs = model.get_covariances()

i, j = np.triu_indices(canonical_covs.shape[-1], k=1)
corr = np.corrcoef(np.concatenate([canonical_covs[:, i, j], fine_tuned_covs[:, i, j]]))[:n_states, n_states:]

print("Correlation of each fine-tuned state with its canonical state:")
print(np.round(np.diag(corr), 3))

fig, ax = plotting.plot_matrices(corr, titles=["Canonical (rows) vs fine-tuned (columns)"])

Finally, we save the fine-tuned model. It can be reloaded with `osl_dynamics.models.load`.

In [ ]:
model.save(f"{output_dir}/model")

## 4. Inference with the Fine-Tuned Model

The state probabilities for each session (in the same order as `files`).

In [ ]:
alp = model.get_alpha(data)
pickle.dump(alp, open(f"{output_dir}/alp.pkl", "wb"))

# Plot the first 8 seconds of the first session
fig, ax = plotting.plot_alpha(alp[0], n_samples=2000, sampling_frequency=250)

Session-specific state covariances can be estimated with the state probabilities held fixed. This is called **dual estimation**. These can be used for group-level statistics, e.g. with `osl_dynamics.stats`.

In [ ]:
means, covs = model.dual_estimation(data, alp, n_jobs=8)
print(covs.shape)

np.save(f"{output_dir}/session_covs.npy", covs)

The post-hoc analysis (spectra, networks, summary statistics) is the same as in the main tutorials, e.g. `elekta_meg_parcel.ipynb`, with `alp` now being a list with one array per session.

#### Alternative: session-specific fine tuning

Instead of fine tuning on the whole dataset, each session can be fine tuned separately, starting from the group-level model every time, with the transition probability matrix held fixed. This returns session-specific state probabilities and covariances:

```python
alp, means, covs = model.fine_tuning(data, n_epochs=2, learning_rate=0.001)
```

This fits each session more closely, but with little data per session the states can drift from the canonical networks, so check the covariances as above. Dual estimation (which only re-estimates the covariances) is the more conservative choice.